<a href="https://colab.research.google.com/github/Sky031826/detox-space-debris/blob/main/notebooks/01_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sgp4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.7/235.7 kB 6.0 MB/s eta 0:00:00


In [2]:
# Download a sample TLE dataset from CelesTrak
# This pulls the "active satellites" catalog — small and reliable for testing
import requests

url = "https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle"
response = requests.get(url)
tle_text = response.text

# Save it locally so we don't keep hitting the API
with open("active_satellites.tle", "w") as f:
    f.write(tle_text)

# Quick check — how many satellites did we get?
lines = tle_text.strip().split('\n')
num_satellites = len(lines) // 3  # each TLE record is 3 lines (name + 2 data lines)
print(f"Downloaded {num_satellites} satellites")
print(f"\nFirst 6 lines (= first 2 satellites):")
for line in lines[:6]:
    print(line)

Downloaded 15380 satellites

First 6 lines (= first 2 satellites):
CALSPHERE 1             
1 00900U 64063C   26130.15989185  .00000547  00000+0  54720-3 0  9992
2 00900  90.2228  70.8741 0028359  49.8747  55.6744 13.76579182 66199
CALSPHERE 2             
1 00902U 64063E   26129.86807183  .00000038  00000+0  43854-4 0  9999
2 00902  90.2360  74.8732 0018054 345.0410  76.0777 13.52897360851089


In [3]:
# Take ISS as our first test case (it's a famous satellite, easy to verify)
from sgp4.api import Satrec, jday

# Find ISS in our downloaded data
lines = tle_text.strip().split('\n')

# Search for ISS
iss_index = None
for i in range(0, len(lines), 3):
    if 'ISS' in lines[i] or 'ZARYA' in lines[i]:
        iss_index = i
        break

if iss_index is not None:
    name = lines[iss_index].strip()
    line1 = lines[iss_index + 1]
    line2 = lines[iss_index + 2]
    print(f"Found: {name}")
    print(f"Line 1: {line1}")
    print(f"Line 2: {line2}")
else:
    # If ISS isn't in 'active' catalog, just use the first satellite
    print("ISS not found, using first satellite instead")
    name = lines[0].strip()
    line1 = lines[1]
    line2 = lines[2]
    print(f"Using: {name}")

Found: ISS (ZARYA)
Line 1: 1 25544U 98067A   26130.19913920  .00006212  00000+0  12008-3 0  9990
Line 2: 2 25544  51.6310 128.1454 0007453  42.8467 317.3101 15.49169801565863


In [4]:
# Create a satellite object from the TLE
satellite = Satrec.twoline2rv(line1, line2)

# Pick a date/time to propagate to: let's say May 15, 2026 at 12:00 UTC
year, month, day = 2026, 5, 15
hour, minute, second = 12, 0, 0

# Convert to Julian date (the format sgp4 uses)
jd, fr = jday(year, month, day, hour, minute, second)

# Propagate the orbit to that time
error_code, position, velocity = satellite.sgp4(jd, fr)

if error_code == 0:
    print(f"Satellite: {name}")
    print(f"Date/time: {year}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}:{second:02d} UTC")
    print(f"\nPosition (km, in ECI frame):")
    print(f"  X: {position[0]:.2f}")
    print(f"  Y: {position[1]:.2f}")
    print(f"  Z: {position[2]:.2f}")
    print(f"\nVelocity (km/s):")
    print(f"  Vx: {velocity[0]:.4f}")
    print(f"  Vy: {velocity[1]:.4f}")
    print(f"  Vz: {velocity[2]:.4f}")

    # Calculate altitude (rough — distance from Earth center minus Earth radius)
    import math
    distance_from_center = math.sqrt(position[0]**2 + position[1]**2 + position[2]**2)
    altitude = distance_from_center - 6371  # Earth radius in km
    print(f"\nAltitude above Earth: {altitude:.2f} km")
else:
    print(f"Propagation error: {error_code}")

Satellite: ISS (ZARYA)
Date/time: 2026-05-15 12:00:00 UTC

Position (km, in ECI frame):
  X: -4312.45
  Y: 2207.70
  Z: 4751.62

Velocity (km/s):
  Vx: -0.6767
  Vy: -7.1460
  Vz: 2.6993

Altitude above Earth: 414.94 km
